# 新栄丸 (SHIN EI MARU / IMO: 9866512) 航跡座標の定期取得・蓄積

VesselFinder の船舶詳細ページから IMO `9866512` の現在位置（緯度・経度・船速・針路・AIS更新時刻など）を定期的に取得し、CSVファイル（`SHIN_EI_MARU_track.csv`）に追記保存します。

In [17]:
import os
import re
import json
import time
from datetime import datetime, timezone, timedelta
import requests
from bs4 import BeautifulSoup
import pandas as pd

# ===== 設定パラメータ =====
IMO = "9866512"
CSV_PATH = "SHIN_EI_MARU_track.csv"
INTERVAL_MINUTES = 10          # 取得間隔（分）
ONLY_SAVE_ON_UPDATE = True     # True: 前回と同じ座標・AIS時刻なら重複保存しない / False: 毎回保存する

JST = timezone(timedelta(hours=9))

In [18]:
def fetch_vessel_position(imo: str = IMO) -> dict:
    """
    VesselFinder の船舶ページから現在座標・船速・針路等を取得して辞書で返す。
    """
    url = f"https://www.vesselfinder.com/vessels/details/{imo}"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
    }

    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    html = resp.text
    soup = BeautifulSoup(html, "html.parser")

    data = {
        "fetched_at_jst": datetime.now(JST).strftime("%Y-%m-%d %H:%M:%S"),
        "imo": imo,
        "mmsi": None,
        "ship_name": None,
        "ais_timestamp_utc": None,
        "latitude": None,
        "longitude": None,
        "sog_kn": None,
        "cog_deg": None,
        "nav_status": None,
    }

    # 1. ページ内の埋め込みJSON (id="djson" 等) からの抽出を試行
    djson_el = soup.find(id="djson")
    if djson_el and djson_el.get("data-json"):
        try:
            vdata = json.loads(djson_el["data-json"])
            data["mmsi"] = vdata.get("mmsi") or data["mmsi"]
            data["ship_name"] = vdata.get("name") or data["ship_name"]
            data["latitude"] = vdata.get("ship_lat") or vdata.get("lat")
            data["longitude"] = vdata.get("ship_lon") or vdata.get("lon")
            data["sog_kn"] = vdata.get("ship_sog") or vdata.get("sog")
            data["cog_deg"] = vdata.get("ship_cog") or vdata.get("cog")
        except json.JSONDecodeError:
            pass

    # 2. テーブル要素 (.aparams 等) からの抽出
    for row in soup.select("table.aparams tr"):
        cells = row.find_all(["td", "th"])
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True).lower()
        val = cells[1].get_text(strip=True)

        if "coordinates" in label and (data["latitude"] is None or data["longitude"] is None):
            # 例: "37.12345 N / 141.12345 E"
            m = re.search(r"([0-9.]+)\s*([NS])\s*/\s*([0-9.]+)\s*([EW])", val, re.I)
            if m:
                lat = float(m.group(1)) * (-1 if m.group(2).upper() == "S" else 1)
                lon = float(m.group(3)) * (-1 if m.group(4).upper() == "W" else 1)
                data["latitude"] = lat
                data["longitude"] = lon
        elif "course / speed" in label:
            # 例: "123.4° / 10.5 kn"
            m = re.search(r"([0-9.]+)\s*°?\s*/\s*([0-9.]+)\s*kn", val, re.I)
            if m:
                if data["cog_deg"] is None:
                    data["cog_deg"] = float(m.group(1))
                if data["sog_kn"] is None:
                    data["sog_kn"] = float(m.group(2))
        elif "navigation status" in label:
            data["nav_status"] = val if val != "-" else None
        elif "imo / mmsi" in label and data["mmsi"] is None:
            m = re.search(r"/\s*(\d{9})", val)
            if m:
                data["mmsi"] = m.group(1)

    # 3. テキスト・メタ情報からのフォールバック抽出
    if data["latitude"] is None or data["longitude"] is None:
        m_coord = re.search(
            r"coordinates\s+([0-9.]+)\s*([NS])\s*/\s*([0-9.]+)\s*([EW])",
            html,
            re.I,
        )
        if m_coord:
            data["latitude"] = float(m_coord.group(1)) * (-1 if m_coord.group(2).upper() == "S" else 1)
            data["longitude"] = float(m_coord.group(3)) * (-1 if m_coord.group(4).upper() == "W" else 1)

    # AIS受信時刻 (id="last_report_ts" または data-timestamp 等)
    ts_el = soup.find(id="last_report_ts") or soup.find(attrs={"data-timestamp": True})
    if ts_el:
        ts_val = ts_el.get("data-timestamp") or ts_el.get_text(strip=True)
        if ts_val and str(ts_val).isdigit():
            dt_utc = datetime.fromtimestamp(int(ts_val), tz=timezone.utc)
            data["ais_timestamp_utc"] = dt_utc.strftime("%Y-%m-%d %H:%M:%S")
        else:
            data["ais_timestamp_utc"] = ts_val

    if data["ship_name"] is None:
        h1 = soup.find("h1")
        if h1:
            data["ship_name"] = h1.get_text(strip=True)

    return data

In [19]:
def append_to_csv(record: dict, csv_path: str = CSV_PATH, only_on_update: bool = ONLY_SAVE_ON_UPDATE) -> bool:
    """
    取得した1件のレコードをCSVに追記する。
    only_on_update=True の場合、直近の行と座標・AIS時刻が同一であればスキップする。
    戻り値: 保存した場合は True、スキップした場合は False
    """
    df_new = pd.DataFrame([record])

    if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
        if only_on_update:
            df_existing = pd.read_csv(csv_path)
            if not df_existing.empty:
                last = df_existing.iloc[-1]
                same_lat = pd.notna(last.get("latitude")) and last.get("latitude") == record.get("latitude")
                same_lon = pd.notna(last.get("longitude")) and last.get("longitude") == record.get("longitude")
                same_ts = str(last.get("ais_timestamp_utc")) == str(record.get("ais_timestamp_utc"))
                if same_lat and same_lon and same_ts:
                    return False
        df_new.to_csv(csv_path, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_new.to_csv(csv_path, mode="w", header=True, index=False, encoding="utf-8-sig")

    return True

In [20]:
# ===== 単発テスト実行 =====
record = fetch_vessel_position(IMO)
saved = append_to_csv(record, CSV_PATH, only_on_update=False)
print(f"Saved: {saved}")
pd.DataFrame([record])

Saved: True


,fetched_at_jst,imo,mmsi,ship_name,ais_timestamp_utc,latitude,longitude,sog_kn,cog_deg,nav_status
0,2026-11-12 15:20:11,9866512,431015938,SHIN EI MARU,None,36,141,20.3,48.9,None


In [ ]:
# ===== 定期実行ループ（停止する時は Jupyter の Interrupt / ■ ボタン） =====
def run_periodic_collector(imo: str = IMO, interval_minutes: float = INTERVAL_MINUTES, csv_path: str = CSV_PATH):
    print(f"IMO {imo} の座標収集を開始します（間隔: {interval_minutes} 分, 保存先: {csv_path}）")
    try:
        while True:
            try:
                rec = fetch_vessel_position(imo)
                is_saved = append_to_csv(rec, csv_path=csv_path, only_on_update=ONLY_SAVE_ON_UPDATE)
                status_str = "追記完了" if is_saved else "更新なし(スキップ)"
                print(
                    f"[{rec['fetched_at_jst']}] {status_str} | "
                    f"Lat: {rec['latitude']}, Lon: {rec['longitude']}, "
                    f"SOG: {rec['sog_kn']} kn, COG: {rec['cog_deg']}°"
                )
            except Exception as e:
                now_str = datetime.now(JST).strftime("%Y-%m-%d %H:%M:%S")
                print(f"[{now_str}] 取得エラー: {e}")

            time.sleep(interval_minutes * 60)
    except KeyboardInterrupt:
        print("\n定期収集を停止しました。")

# 実行する場合は以下のコメントアウトを外すかそのまま実行してください
run_periodic_collector()

IMO 9866512 の座標収集を開始します（間隔: 10 分, 保存先: SHIN_EI_MARU_track.csv）
[2026-11-12 15:20:12] 追記完了 | Lat: 36, Lon: 141, SOG: 20.3 kn, COG: 48.9°


In [ ]:
# ===== 蓄積されたCSVデータの確認 =====
if os.path.exists(CSV_PATH):
    df_track = pd.read_csv(CSV_PATH)
    display(df_track.tail(10))
else:
    print(f"{CSV_PATH} はまだ作成されていません。")

,fetched_at_jst,imo,mmsi,ship_name,ais_timestamp_utc,latitude,longitude,sog_kn,cog_deg,nav_status
1,2026-11-12 15:02:16,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
2,2026-11-12 15:05:56,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
3,2026-11-12 15:07:31,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
4,2026-11-12 15:07:32,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
5,2026-11-12 15:16:52,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
6,2026-11-12 15:16:53,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
7,2026-11-12 15:19:41,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
8,2026-11-12 15:19:43,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
9,2026-11-12 15:19:50,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
10,2026-11-12 15:19:51,9866512,431015938,SHIN EI MARU,NaN,36,141,20.3,48.9,NaN
